# Vera CoT Verifier Fine-Tuning

Fine-tune LFM2.5-1.2B on claim verification with reasoning using Unsloth + LoRA.

**Sections:**
1. Setup & Installation
2. Load Model
3. Data Preparation
4. Training
5. Save Model
6. Inference
7. Evaluation (Fine-tuned, Zero-shot, Few-shot)
8. Comparison Table

## 1. Setup & Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2
!pip install sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = 'drive/MyDrive/vera/verification'
TRAIN_FILE = f'{DATA_DIR}/vera_train.jsonl'
VAL_FILE = f'{DATA_DIR}/vera_val.jsonl'
TEST_FILE = f'{DATA_DIR}/vera_test.jsonl'
OUTPUT_DIR = f'{DATA_DIR}/model_output'

Mounted at /content/drive


## 2. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


## 3. Data Preparation

In [ ]:
import json
from datasets import Dataset

def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

train_data = load_jsonl(TRAIN_FILE)
val_data = load_jsonl(VAL_FILE)
test_data = load_jsonl(TEST_FILE)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Train: 1708, Val: 376, Test: 373


In [ ]:
STUDENT_PROMPT = """You are a fact-checker. Verify claims against evidence.

TASK: Given a CLAIM and EVIDENCE snippets, determine if the claim is supported or refuted.

OUTPUT FORMAT (JSON):
{
  "reasoning": "<analyze each evidence piece, then conclude>",
  "verdict": "SUPPORTED" | "REFUTED"
}

RULES:
1. SUPPORTED = claim confirmed by evidence
2. REFUTED = evidence contradicts the claim
3. Reference evidence by number: "Evidence 1 shows..."
4. If any key part is contradicted, verdict is REFUTED
"""

TEACHER_PROMPT = """# ROLE
You are an expert fact-checker. Verify claims against provided evidence and produce a reasoned verdict.

Given a CLAIM and multiple EVIDENCE snippets, you must:

1. **ANALYZE** each piece of evidence:
   - Identify what facts each evidence establishes
   - Note if it supports, refutes, or is irrelevant to the claim

2. **REASON** step-by-step:
   - Compare the claim against ALL evidence pieces
   - Consider partial matches and nuances

3. **VERDICT**:
   - SUPPORTED: The claim is confirmed by evidence
   - REFUTED: Evidence contradicts the claim

{
  "reasoning": "<step-by-step analysis of each evidence vs claim>",
  "verdict": "SUPPORTED" | "REFUTED"
}

- Cite specific evidence in your reasoning (e.g., "Evidence 1 states...")
- Consider ALL evidence before reaching a verdict
- If any key part of the claim is contradicted, verdict is REFUTED


CLAIM: "Donald Trump delivered the largest tax cuts in American history."

EVIDENCE:
[1] This tax cut is the 8th largest as a percent of GDP since 1918 and the 4th largest in inflation-adjusted dollars.
[2] Three tax bills have been larger: American Taxpayer Relief Act of 2012, Tax Relief Act of 2010, and Economic Recovery Tax Act of 1981.

OUTPUT:
{
  "reasoning": "The claim states Trump delivered the 'largest' tax cuts. Evidence 1 shows it ranks 8th by GDP and 4th in dollars. Evidence 2 names three larger bills. Both contradict 'largest' - the claim is factually incorrect.",
  "verdict": "REFUTED"
}

CLAIM: "Biden has pledged to stop border wall construction."

EVIDENCE:
[1] "No, there will not be another foot of wall constructed on my administration... I'm going to make sure that we have border protection, but it's going to be based on making sure that we use high-tech capacity."

OUTPUT:
{
  "reasoning": "Evidence 1 directly quotes Biden pledging 'there will not be another foot of wall constructed.' This confirms the claim.",
  "verdict": "SUPPORTED"
}

CLAIM: "The salaries of MPs in the UK have grown faster than starting salaries of nurses, teachers and police officers between 2010 and 2020."

EVIDENCE:
[1] MPs earned £65,738 in 2010 and £79,468 in 2019 - a 21% increase.
[2] Police constables starting salary increased from £23,259 to £26,199 - a 13% increase.
[3] Teachers starting salary increased from £21,588 to £24,373 - a 13% increase.
[4] Nurses starting salary increased from £21,176 to £24,214 - a 14% increase.

OUTPUT:
{
  "reasoning": "Evidence 1 shows MPs had 21% salary growth. Evidence 2-4 show nurses (14%), teachers (13%), and police (13%) all had lower growth. MPs' 21% exceeds all three professions.",
  "verdict": "SUPPORTED"
}
"""

In [ ]:
LABEL_MAP = {
    "Supported": "SUPPORTED",
    "Refuted": "REFUTED",
}

def format_input(claim, evidence):
    """Format claim and evidence for the model."""
    lines = [f"CLAIM: {claim}", "", "EVIDENCE:"]
    for i, e in enumerate(evidence, 1):
        lines.append(f"[{i}] {e}")
    return "\n".join(lines)

def format_sample(sample):
    """Convert sample to chat format."""
    claim = sample['input']['claim']
    evidence = sample['input']['evidence']
    output = sample['output']

    input_text = format_input(claim, evidence)
    output_json = json.dumps(output, ensure_ascii=False)

    conversations = [
        {"role": "system", "content": STUDENT_PROMPT},
        {"role": "user", "content": input_text},
        {"role": "assistant", "content": output_json}
    ]
    return {"conversations": conversations}

train_formatted = [format_sample(s) for s in train_data]
val_formatted = [format_sample(s) for s in val_data]

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 1708, Val: 376


In [ ]:
from collections import Counter

def count_labels(data, name):
    labels = [LABEL_MAP.get(s['ground_truth'], s['output']['verdict']) for s in data]
    counts = Counter(labels)
    print(f"{name}:")
    print(f"  SUPPORTED: {counts.get('SUPPORTED', 0)}")
    print(f"  REFUTED:   {counts.get('REFUTED', 0)}")
    print(f"  Total:     {len(data)}")
    return counts

print("=" * 40)
print("Label Distribution")
print("=" * 40)
count_labels(train_data, "Train")
count_labels(val_data, "Val")
count_labels(test_data, "Test")

Label Distribution
Train:
  SUPPORTED: 507
  REFUTED:   1201
  Total:     1708
Val:
  SUPPORTED: 99
  REFUTED:   277
  Total:     376
Test:
  SUPPORTED: 115
  REFUTED:   258
  Total:     373


Counter({'REFUTED': 258, 'SUPPORTED': 115})

In [ ]:
import random

train_supported = []
train_refuted = []

for s in train_data:
    raw_label = s.get('ground_truth') or s.get('output', {}).get('verdict')
    norm_label = LABEL_MAP.get(raw_label, raw_label)

    if norm_label == "SUPPORTED":
        train_supported.append(s)
    elif norm_label == "REFUTED":
        train_refuted.append(s)

imbalance_gap = len(train_refuted) - len(train_supported)

if imbalance_gap > 0:
    print(f"Oversampling SUPPORTED by {imbalance_gap} samples...")
    upsampled_data = random.choices(train_supported, k=imbalance_gap)
    train_data = train_supported + train_refuted + upsampled_data
    random.shuffle(train_data)
    print(f"New Balanced Train Size: {len(train_data)}")

Oversampling SUPPORTED by 591 samples...
New Balanced Train Size: 2174


In [ ]:

print("=" * 40)
print("Label Distribution")
print("=" * 40)
count_labels(train_data, "Train")
count_labels(val_data, "Val")
count_labels(test_data, "Test")

Label Distribution
Train:
  SUPPORTED: 1087
  REFUTED:   1087
  Total:     2174
Val:
  SUPPORTED: 99
  REFUTED:   277
  Total:     376
Test:
  SUPPORTED: 115
  REFUTED:   258
  Total:     373


Counter({'REFUTED': 258, 'SUPPORTED': 115})

In [ ]:
def formatting_prompts_func(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("Sample:")
print(train_dataset[0]["text"][:800])

Map:   0%|          | 0/1708 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

Sample:
<|im_start|>system
You are a fact-checker. Verify claims against evidence.

TASK: Given a CLAIM and EVIDENCE snippets, determine if the claim is supported or refuted.

OUTPUT FORMAT (JSON):
{
  "reasoning": "<analyze each evidence piece, then conclude>",
  "verdict": "SUPPORTED" | "REFUTED"
}

RULES:
1. SUPPORTED = claim confirmed by evidence
2. REFUTED = evidence contradicts the claim
3. Reference evidence by number: "Evidence 1 shows..."
4. If any key part is contradicted, verdict is REFUTED
<|im_end|>
<|im_start|>user
CLAIM: US Senator Amy Klobuchar vows to resettle no refugees in her neighbourhood.

EVIDENCE:
[1] Sen. Amy Klobuchar (D-MN) vowed to increase the number of refugees resettled in the United States by more than 500 percent if elected president.
[2] Yes. Sen. Amy Klobuchar (D


## 4. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        per_device_eval_batch_size = 16,
        gradient_accumulation_steps = 1,
        warmup_steps = 50,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1708 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/376 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=16):   0%|          | 0/1708 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/376 [00:00<?, ? examples/s]

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
2.23 GB reserved.


In [ ]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,708 | Num Epochs = 3 | Total steps = 642
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 11,108,352 of 1,181,448,960 (0.94% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.741800,0.736714
2,0.567500,0.726318
3,0.418900,0.785929


## 5. Save Model

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapters")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapters")
print(f"Saved to {OUTPUT_DIR}/lora_adapters")

Saved to drive/MyDrive/vera/verification/model_output/lora_adapters


## 6. Inference

In [ ]:
FastLanguageModel.for_inference(model)

def verify_claim(claim, evidence, system_prompt, max_new_tokens=512):
    """Run verification with given system prompt."""
    input_text = format_input(claim, evidence)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.1,
        do_sample=True,
    )

    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

In [ ]:
test_claim = "The Eiffel Tower is located in London."
test_evidence = ["The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France."]

print("Claim:", test_claim)
print("Evidence:", test_evidence)
print("\nOutput (fine-tuned):")
print(verify_claim(test_claim, test_evidence, STUDENT_PROMPT))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Claim: The Eiffel Tower is located in London.
Evidence: ['The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.']

Output (fine-tuned):
{"reasoning": "The claim asserts that the Eiffel Tower is located in London. Evidence 1 explicitly states that the Eiffel Tower is located in Paris, France. Since the evidence directly contradicts the location mentioned in the claim, the claim is false.", "verdict": "REFUTED"}


## 7. Evaluation

Three evaluations:
1. **Fine-tuned** - Our model with student prompt
2. **Zero-shot** - Base model with student prompt (no examples)
3. **Few-shot** - Base model with teacher prompt (has examples)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

EVAL_BATCH_SIZE = 8

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def parse_json_safe(text):
    """Parse JSON from model output. Returns (parsed_dict, is_valid_json)."""
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed, True
        return {}, False
    except (json.JSONDecodeError, ValueError, TypeError):
        try:
            start = text.find('{')
            end = text.rfind('}') + 1
            if start >= 0 and end > start:
                parsed = json.loads(text[start:end])
                return parsed, True
        except:
            pass
    return {}, False


def compute_metrics(predictions, ground_truths):
    """Compute accuracy, precision, recall, F1 for binary classification."""
    if not predictions or not ground_truths:
        return {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0}

    y_pred = [1 if p == "SUPPORTED" else 0 for p in predictions]
    y_true = [1 if g == "SUPPORTED" else 0 for g in ground_truths]

    correct = sum(1 for p, g in zip(predictions, ground_truths) if p == g)
    accuracy = correct / len(predictions)

    tp = sum(1 for p, g in zip(y_pred, y_true) if p == 1 and g == 1)
    fp = sum(1 for p, g in zip(y_pred, y_true) if p == 1 and g == 0)
    fn = sum(1 for p, g in zip(y_pred, y_true) if p == 0 and g == 1)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4)
    }

In [ ]:
def batched_generate(samples, system_prompt, curr_model, curr_tokenizer, batch_size=8, max_new_tokens=512):
    all_outputs = []

    curr_tokenizer.padding_side = "left"
    if not curr_tokenizer.pad_token:
        curr_tokenizer.pad_token = curr_tokenizer.eos_token

    for i in range(0, len(samples), batch_size):
        batch = samples[i:i+batch_size]

        batch_messages = [
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": format_input(s['input']['claim'], s['input']['evidence'])}
            ]
            for s in batch
        ]

        prompts = [
            curr_tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            for msg in batch_messages
        ]

        inputs = curr_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to("cuda")

        outputs = curr_model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            temperature=0.1,
            do_sample=True,
            pad_token_id=curr_tokenizer.pad_token_id,
            stop_strings=["}", "}\n"],
            tokenizer=curr_tokenizer
        )

        prompt_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[:, prompt_length:]
        decoded_batch = curr_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        all_outputs.extend(decoded_batch)

    return all_outputs

In [ ]:
def evaluate_batched(test_data, system_prompt, curr_model, curr_tokenizer, max_samples=100):
    """Evaluate model with batched inference."""
    samples = test_data[:max_samples]

    print(f"  Generating {len(samples)} outputs in batches of {EVAL_BATCH_SIZE}...")
    pred_outputs = batched_generate(samples, system_prompt, curr_model, curr_tokenizer, batch_size=EVAL_BATCH_SIZE)

    predictions = []
    ground_truths = []
    json_valid = 0
    json_invalid = 0

    for pred_output, sample in zip(pred_outputs, samples):
        parsed, is_valid = parse_json_safe(pred_output)

        if is_valid and 'verdict' in parsed:
            json_valid += 1
            pred_verdict = parsed['verdict']
        else:
            json_invalid += 1
            if 'SUPPORTED' in pred_output.upper():
                pred_verdict = 'SUPPORTED'
            elif 'REFUTED' in pred_output.upper():
                pred_verdict = 'REFUTED'
            else:
                pred_verdict = 'REFUTED'  # Default

        predictions.append(pred_verdict)
        gt_verdict = LABEL_MAP.get(sample['ground_truth'], sample['output']['verdict'])
        ground_truths.append(gt_verdict)

    metrics = compute_metrics(predictions, ground_truths)
    metrics['json_valid_rate'] = round(json_valid / len(samples), 4)

    return metrics, predictions, ground_truths

In [ ]:
print("=" * 50)
print("Evaluating FINE-TUNED model (student prompt)...")
print("=" * 50)
finetuned_metrics, ft_preds, ft_gts = evaluate_batched(test_data, STUDENT_PROMPT, model, tokenizer, max_samples=len(test_data))
print(f"Fine-tuned: {finetuned_metrics}")

Evaluating FINE-TUNED model (student prompt)...
  Generating 373 outputs in batches of 8...
Fine-tuned: {'accuracy': 0.8928, 'precision': 0.9032, 'recall': 0.7304, 'f1': 0.8077, 'json_valid_rate': 1.0}


In [ ]:
print("\nLoading base model for baselines...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
FastLanguageModel.for_inference(base_model)


Loading base model for baselines...
==((====))==  Unsloth 2026.1.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(2048, 2048, kernel_size=(3,), stride=(1,), padding=(2,), groups=2048, bias=False)
          (in_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear(in_features=2048, out_features=8192, bias=False)
          (w3): Linear(in_features=2048, out_features=8192, bias=False)
          (w2): Linear(in_features=8192, out_features=2048, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((2048,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((2048,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Li

In [ ]:
print("\n" + "=" * 50)
print("Evaluating ZERO-SHOT (base model + student prompt)...")
print("=" * 50)
zeroshot_metrics, zs_preds, zs_gts = evaluate_batched(test_data, STUDENT_PROMPT, base_model, base_tokenizer, max_samples=len(test_data))
print(f"Zero-shot: {zeroshot_metrics}")


Evaluating ZERO-SHOT (base model + student prompt)...
  Generating 373 outputs in batches of 8...
Zero-shot: {'accuracy': 0.7802, 'precision': 0.646, 'recall': 0.6348, 'f1': 0.6404, 'json_valid_rate': 1.0}


In [ ]:
print("\n" + "=" * 50)
print("Evaluating FEW-SHOT (base model + teacher prompt with examples)...")
print("=" * 50)
fewshot_metrics, fs_preds, fs_gts = evaluate_batched(test_data, TEACHER_PROMPT, base_model, base_tokenizer, max_samples=len(test_data))
print(f"Few-shot: {fewshot_metrics}")


Evaluating FEW-SHOT (base model + teacher prompt with examples)...
  Generating 373 outputs in batches of 8...
Few-shot: {'accuracy': 0.7534, 'precision': 0.5689, 'recall': 0.8261, 'f1': 0.6738, 'json_valid_rate': 0.6944}


## 8. Comparison Table

In [ ]:
import pandas as pd

results = {
    "Model": ["Fine-tuned", "Zero-shot", "Few-shot"],
    "Accuracy": [finetuned_metrics['accuracy'], zeroshot_metrics['accuracy'], fewshot_metrics['accuracy']],
    "Precision": [finetuned_metrics['precision'], zeroshot_metrics['precision'], fewshot_metrics['precision']],
    "Recall": [finetuned_metrics['recall'], zeroshot_metrics['recall'], fewshot_metrics['recall']],
    "F1": [finetuned_metrics['f1'], zeroshot_metrics['f1'], fewshot_metrics['f1']],
    "JSON Valid %": [finetuned_metrics['json_valid_rate'], zeroshot_metrics['json_valid_rate'], fewshot_metrics['json_valid_rate']],
}

df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("FINAL COMPARISON")
print("=" * 60)
display(df)


FINAL COMPARISON


,Model,Accuracy,Precision,Recall,F1,JSON Valid %
0,Fine-tuned,0.8928,0.9032,0.7304,0.8077,1.0000
1,Zero-shot,0.7802,0.6460,0.6348,0.6404,1.0000
2,Few-shot,0.7534,0.5689,0.8261,0.6738,0.6944


In [ ]:
results_path = f"{OUTPUT_DIR}/evaluation_results.json"
with open(results_path, 'w') as f:
    json.dump({
        "finetuned": finetuned_metrics,
        "zeroshot": zeroshot_metrics,
        "fewshot": fewshot_metrics
    }, f, indent=2)
print(f"Saved results to {results_path}")

Saved results to drive/MyDrive/vera/verification/model_output/evaluation_results.json


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"{OUTPUT_DIR}/lora_adapters",
    max_seq_length = 2048,
)

model.push_to_hub_gguf(
    "werstal/vera-verifier-cot-lfm2.5-1.2b-gguf",
    tokenizer,
    quantization_method = "f16",
    token = None  # Add hf token here
)

print("Saved GGUF f16 to HuggingFace!")

==((====))==  Unsloth 2026.1.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/tmp/unsloth_gguf_sgvgoox9`: 100%|██████████| 1/1 [00:04<00:00,  4.66s/it]


Successfully copied all 1 files from cache to `/tmp/unsloth_gguf_sgvgoox9`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:10<00:00, 10.05s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_sgvgoox9`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['LFM2.5-1.2B-Instruct.BF16.gguf']


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ....5-1.2B-Instruct.F16.gguf:   1%|1         | 31.2MB / 2.34GB            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/werstal/vera-verifier-cot-lfm2.5-1.2b-gguf
Unsloth: Cleaning up temporary files...
Saved GGUF f16 to HuggingFace!
